In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rhythmbhetwal/ecg-data/mitbih_test.csv
/kaggle/input/datasets/rhythmbhetwal/ecg-data/mitbih_train.csv


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input, LSTM, Bidirectional, Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Layer


2026-03-05 06:15:22.490584: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772691322.690703      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772691322.744727      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772691323.177974      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772691323.178014      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772691323.178017      24 computation_placer.cc:177] computation placer alr

In [3]:
train_df = pd.read_csv("/kaggle/input/datasets/rhythmbhetwal/ecg-data/mitbih_train.csv", header=None)
test_df  = pd.read_csv("/kaggle/input/datasets/rhythmbhetwal/ecg-data/mitbih_test.csv", header=None)

In [4]:
X = train_df.iloc[:, :-1].values
y = train_df.iloc[:, -1].values


In [5]:
X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values


In [6]:
print("Train shape:", X.shape)
print("Test shape:", X_test.shape)



Train shape: (87554, 187)
Test shape: (21892, 187)


In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [8]:
def moving_average(signal, window=5):
    return np.convolve(signal, np.ones(window)/window, mode="same")

In [9]:
def create_ts_channels(signal):

    original = signal
    smooth = moving_average(signal)
    derivative = np.gradient(signal)

    def norm(x):
        return (x - np.mean(x)) / (np.std(x) + 1e-8)

    return np.stack([
        norm(original),
        norm(smooth),
        norm(derivative)
    ], axis=1)


In [10]:
X_train_ts = np.array([create_ts_channels(x) for x in X_train])
X_val_ts   = np.array([create_ts_channels(x) for x in X_val])
X_test_ts  = np.array([create_ts_channels(x) for x in X_test])
print("Processed train shape:", X_train_ts.shape)


Processed train shape: (70043, 187, 3)


In [11]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

In [12]:
class_weight_dict = {
    int(c): float(w ** 0.5)
    for c, w in zip(np.unique(y_train), class_weights)
}

print(class_weight_dict)

{0: 0.49155203425853494, 1: 2.8069293976549488, 2: 1.7394296624916856, 3: 5.2256303424212405, 4: 1.6500787742187668}


In [13]:
class TemporalAttention(Layer):

    def build(self, input_shape):

        self.W = self.add_weight(
            shape=(input_shape[-1],1),
            initializer="normal"
        )

        self.b = self.add_weight(
            shape=(input_shape[1],1),
            initializer="zeros"
        )

    def call(self,x):

        e = tf.tanh(tf.matmul(x,self.W) + self.b)

        a = tf.nn.softmax(e, axis=1)

        output = tf.reduce_sum(x * a, axis=1)

        return output


In [14]:
def build_attention_model():

    inputs = Input(shape=(187,3))

    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    x = Dropout(0.3)(x)

    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Dropout(0.3)(x)

    x = TemporalAttention()(x)

    x = Dense(64, activation="relu")(x)
    outputs = Dense(5, activation="softmax")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [15]:
def build_no_attention_model():

    inputs = Input(shape=(187,3))

    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    x = Dropout(0.3)(x)

    x = Bidirectional(LSTM(64))(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation="relu")(x)
    outputs = Dense(5, activation="softmax")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [16]:
def build_raw_model():

    inputs = Input(shape=(187,1))

    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    x = Dropout(0.3)(x)

    x = Bidirectional(LSTM(64))(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation="relu")(x)
    outputs = Dense(5, activation="softmax")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [17]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

def evaluate(model, X_val, y_val, name):

    y_pred = np.argmax(model.predict(X_val), axis=1)

    acc = accuracy_score(y_val, y_pred)
    f1  = f1_score(y_val, y_pred, average="macro")

    print("\n==========", name, "==========")
    print("Accuracy:", acc)
    print("Macro F1:", f1)

    print("\nClassification Report:")
    print(classification_report(y_val, y_pred))

    return acc, f1

In [18]:
model_full = build_attention_model()

model_full.fit(
    X_train_ts,
    y_train,
    validation_data=(X_val_ts,y_val),
    epochs=20,
    batch_size=128,
    class_weight=class_weight_dict
)

acc_full, f1_full = evaluate(model_full, X_val_ts, y_val, "FULL MODEL")

I0000 00:00:1772691373.134103      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/20


I0000 00:00:1772691379.418788      64 cuda_dnn.cc:529] Loaded cuDNN version 91002


548/548 ━━━━━━━━━━━━━━━━━━━━ 36s 54ms/step - accuracy: 0.8585 - loss: 0.6694 - val_accuracy: 0.8902 - val_loss: 0.3828
Epoch 2/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 29s 53ms/step - accuracy: 0.9353 - loss: 0.2775 - val_accuracy: 0.9326 - val_loss: 0.2325
Epoch 3/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 29s 53ms/step - accuracy: 0.9485 - loss: 0.2191 - val_accuracy: 0.9612 - val_loss: 0.1433
Epoch 4/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 29s 53ms/step - accuracy: 0.9563 - loss: 0.1849 - val_accuracy: 0.9525 - val_loss: 0.1699
Epoch 5/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 29s 53ms/step - accuracy: 0.9624 - loss: 0.1590 - val_accuracy: 0.9616 - val_loss: 0.1323
Epoch 6/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 29s 53ms/step - accuracy: 0.9658 - loss: 0.1423 - val_accuracy: 0.9612 - val_loss: 0.1405
Epoch 7/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 29s 52ms/step - accuracy: 0.9658 - loss: 0.1370 - val_accuracy: 0.9685 - val_loss: 0.1079
Epoch 8/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 29s 53ms/step - accuracy: 0.9728 - loss: 0.1107 - val_accurac

In [19]:
model_no_att = build_no_attention_model()

model_no_att.fit(
    X_train_ts,
    y_train,
    epochs=20,
    batch_size=128
)

acc_no_att, f1_no_att = evaluate(
    model_no_att,
    X_val_ts,
    y_val,
    "NO ATTENTION"
)


Epoch 1/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.8841 - loss: 0.4412
Epoch 2/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9544 - loss: 0.1749
Epoch 3/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9655 - loss: 0.1306
Epoch 4/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9700 - loss: 0.1128
Epoch 5/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9742 - loss: 0.0944
Epoch 6/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9754 - loss: 0.0906
Epoch 7/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9773 - loss: 0.0857
Epoch 8/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9784 - loss: 0.0777
Epoch 9/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9804 - loss: 0.0713
Epoch 10/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9800 - loss: 0.0729
Epoch 11/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9830 - loss: 0.0613
Epoch 12/20
548/548 ━━━━━━━━━━

In [20]:
X_train_raw = X_train.reshape(-1,187,1)
X_val_raw   = X_val.reshape(-1,187,1)

model_raw = build_raw_model()

model_raw.fit(
    X_train_raw,
    y_train,
    epochs=20,
    batch_size=128
)

acc_raw, f1_raw = evaluate(
    model_raw,
    X_val_raw,
    y_val,
    "RAW ECG ONLY"
)


Epoch 1/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.8352 - loss: 0.6280
Epoch 2/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9397 - loss: 0.2289
Epoch 3/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9465 - loss: 0.1946
Epoch 4/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9582 - loss: 0.1517
Epoch 5/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9559 - loss: 0.1664
Epoch 6/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9631 - loss: 0.1404
Epoch 7/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9659 - loss: 0.1240
Epoch 8/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9678 - loss: 0.1188
Epoch 9/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9698 - loss: 0.1102
Epoch 10/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9706 - loss: 0.1091
Epoch 11/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9725 - loss: 0.0980
Epoch 12/20
548/548 ━━━━━━━━━━

In [21]:
model_no_weights = build_attention_model()

model_no_weights.fit(
    X_train_ts,
    y_train,
    epochs=20,
    batch_size=128
)

acc_no_weights, f1_no_weights = evaluate(
    model_no_weights,
    X_val_ts,
    y_val,
    "NO CLASS WEIGHTS"
)

Epoch 1/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.8643 - loss: 0.5299
Epoch 2/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9457 - loss: 0.1999
Epoch 3/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9623 - loss: 0.1423
Epoch 4/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9677 - loss: 0.1194
Epoch 5/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9678 - loss: 0.1176
Epoch 6/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9753 - loss: 0.0907
Epoch 7/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9780 - loss: 0.0779
Epoch 8/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9778 - loss: 0.0777
Epoch 9/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9794 - loss: 0.0743
Epoch 10/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9823 - loss: 0.0634
Epoch 11/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9803 - loss: 0.0690
Epoch 12/20
548/548 ━━━━━━━━━━

In [22]:
results = pd.DataFrame({

    "Model":[
        "Full Model",
        "No Attention",
        "Raw ECG Only",
        "No Class Weights"
    ],

    "Accuracy":[
        acc_full,
        acc_no_att,
        acc_raw,
        acc_no_weights
    ],

    "Macro F1":[
        f1_full,
        f1_no_att,
        f1_raw,
        f1_no_weights
    ]
})

print("\n===== ABLATION RESULTS =====")
print(results)


===== ABLATION RESULTS =====
              Model  Accuracy  Macro F1
0        Full Model  0.979441  0.881416
1      No Attention  0.984524  0.915950
2      Raw ECG Only  0.977842  0.882140
3  No Class Weights  0.984981  0.910388


In [23]:
def create_ts_channels_no_derivative(signal):

    original = signal
    smooth = moving_average(signal)

    def norm(x):
        return (x - np.mean(x)) / (np.std(x) + 1e-8)

    return np.stack([
        norm(original),
        norm(smooth)
    ], axis=1)

In [24]:
X_train_rs = np.array([create_ts_channels_no_derivative(x) for x in X_train])
X_val_rs   = np.array([create_ts_channels_no_derivative(x) for x in X_val])
X_test_rs  = np.array([create_ts_channels_no_derivative(x) for x in X_test])

print("Raw + Smooth shape:", X_train_rs.shape)

Raw + Smooth shape: (70043, 187, 2)


In [25]:
def build_raw_smooth_model():

    inputs = Input(shape=(187,2))

    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    x = Dropout(0.3)(x)

    x = Bidirectional(LSTM(64))(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation="relu")(x)
    outputs = Dense(5, activation="softmax")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [26]:
model_raw_smooth = build_raw_smooth_model()

model_raw_smooth.fit(
    X_train_rs,
    y_train,
    epochs=20,
    batch_size=128
)

Epoch 1/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 28s 46ms/step - accuracy: 0.8764 - loss: 0.4794
Epoch 2/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 46ms/step - accuracy: 0.9492 - loss: 0.1952
Epoch 3/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 46ms/step - accuracy: 0.9559 - loss: 0.1668
Epoch 4/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 46ms/step - accuracy: 0.9590 - loss: 0.1441
Epoch 5/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 46ms/step - accuracy: 0.9660 - loss: 0.1246
Epoch 6/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 46ms/step - accuracy: 0.9676 - loss: 0.1170
Epoch 7/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9692 - loss: 0.1078
Epoch 8/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9725 - loss: 0.0981
Epoch 9/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 46ms/step - accuracy: 0.9756 - loss: 0.0884
Epoch 10/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9757 - loss: 0.0869
Epoch 11/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9791 - loss: 0.0753
Epoch 12/20
548/548 ━━━━━━━━━━

In [27]:
acc_rs, f1_rs = evaluate(
    model_raw_smooth,
    X_val_rs,
    y_val,
    "RAW + SMOOTH (NO DERIVATIVE)"
)

548/548 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step

========== RAW + SMOOTH (NO DERIVATIVE) ==========
Accuracy: 0.9837245160185026
Macro F1: 0.9039331496866134

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     14494
         1.0       0.89      0.79      0.84       445
         2.0       0.95      0.96      0.95      1158
         3.0       0.96      0.61      0.75       128
         4.0       0.99      0.99      0.99      1286

    accuracy                           0.98     17511
   macro avg       0.96      0.87      0.90     17511
weighted avg       0.98      0.98      0.98     17511



In [28]:
results = pd.DataFrame({

    "Model":[
        "Full Model (Raw+Smooth+Derivative)",
        "Raw + Smooth",
        "Raw ECG Only",
        "No Attention",
        "No Class Weights"
    ],

    "Accuracy":[
        acc_full,
        acc_rs,
        acc_raw,
        acc_no_att,
        acc_no_weights
    ],

    "Macro F1":[
        f1_full,
        f1_rs,
        f1_raw,
        f1_no_att,
        f1_no_weights
    ]
})

print("\n===== ABLATION RESULTS =====")
print(results)


===== ABLATION RESULTS =====
                                Model  Accuracy  Macro F1
0  Full Model (Raw+Smooth+Derivative)  0.979441  0.881416
1                        Raw + Smooth  0.983725  0.903933
2                        Raw ECG Only  0.977842  0.882140
3                        No Attention  0.984524  0.915950
4                    No Class Weights  0.984981  0.910388


In [29]:
def build_rs_model():

    inputs = Input(shape=(187,2))

    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    x = Dropout(0.3)(x)

    x = Bidirectional(LSTM(64))(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation="relu")(x)
    outputs = Dense(5, activation="softmax")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [30]:
def create_raw(signal):

    def norm(x):
        return (x - np.mean(x)) / (np.std(x) + 1e-8)

    return norm(signal).reshape(187,1)

In [31]:
X_train_raw = np.array([create_raw(x) for x in X_train])
X_val_raw   = np.array([create_raw(x) for x in X_val])

In [32]:
model_raw = build_raw_model()

model_raw.fit(
    X_train_raw,
    y_train,
    epochs=20,
    batch_size=128
)

acc_raw, f1_raw = evaluate(model_raw, X_val_raw, y_val, "RAW ONLY")

Epoch 1/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.8659 - loss: 0.5030
Epoch 2/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9481 - loss: 0.2024
Epoch 3/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9551 - loss: 0.1656
Epoch 4/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9606 - loss: 0.1440
Epoch 5/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9663 - loss: 0.1248
Epoch 6/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9686 - loss: 0.1150
Epoch 7/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9714 - loss: 0.1056
Epoch 8/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9726 - loss: 0.0975
Epoch 9/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9743 - loss: 0.0935
Epoch 10/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9758 - loss: 0.0891
Epoch 11/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9755 - loss: 0.0861
Epoch 12/20
548/548 ━━━━━━━━━━

In [33]:
def create_raw_smooth(signal):

    smooth = moving_average(signal)

    def norm(x):
        return (x - np.mean(x)) / (np.std(x) + 1e-8)

    return np.stack([
        norm(signal),
        norm(smooth)
    ], axis=1)

In [34]:
X_train_rs = np.array([create_raw_smooth(x) for x in X_train])
X_val_rs   = np.array([create_raw_smooth(x) for x in X_val])

In [35]:
model_rs = build_rs_model()

model_rs.fit(
    X_train_rs,
    y_train,
    epochs=20,
    batch_size=128
)

acc_rs, f1_rs = evaluate(model_rs, X_val_rs, y_val, "RAW + SMOOTH")

Epoch 1/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.8800 - loss: 0.4642
Epoch 2/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9517 - loss: 0.1848
Epoch 3/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9579 - loss: 0.1523
Epoch 4/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9638 - loss: 0.1319
Epoch 5/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9685 - loss: 0.1167
Epoch 6/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9696 - loss: 0.1090
Epoch 7/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9731 - loss: 0.0971
Epoch 8/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9745 - loss: 0.0906
Epoch 9/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9749 - loss: 0.0897
Epoch 10/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9787 - loss: 0.0751
Epoch 11/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9783 - loss: 0.0764
Epoch 12/20
548/548 ━━━━━━━━━━

In [36]:
def create_raw_derivative(signal):

    derivative = np.gradient(signal)

    def norm(x):
        return (x - np.mean(x)) / (np.std(x) + 1e-8)

    return np.stack([
        norm(signal),
        norm(derivative)
    ], axis=1)

In [37]:
X_train_rd = np.array([create_raw_derivative(x) for x in X_train])
X_val_rd   = np.array([create_raw_derivative(x) for x in X_val])

In [38]:
model_rd = build_rs_model()

model_rd.fit(
    X_train_rd,
    y_train,
    epochs=20,
    batch_size=128
)

acc_rd, f1_rd = evaluate(model_rd, X_val_rd, y_val, "RAW + DERIVATIVE")

Epoch 1/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.8881 - loss: 0.4432
Epoch 2/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9521 - loss: 0.1786
Epoch 3/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9646 - loss: 0.1389
Epoch 4/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9674 - loss: 0.1214
Epoch 5/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9729 - loss: 0.0985
Epoch 6/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9766 - loss: 0.0873
Epoch 7/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9780 - loss: 0.0806
Epoch 8/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9795 - loss: 0.0745
Epoch 9/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9790 - loss: 0.0729
Epoch 10/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9819 - loss: 0.0656
Epoch 11/20
548/548 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.9830 - loss: 0.0598
Epoch 12/20
548/548 ━━━━━━━━━━

In [39]:
acc_full, f1_full = evaluate(model_full, X_val_ts, y_val, "FULL MODEL")

548/548 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step

========== FULL MODEL ==========
Accuracy: 0.9794414939181086
Macro F1: 0.8814161621888317

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     14494
         1.0       0.86      0.81      0.84       445
         2.0       0.96      0.93      0.95      1158
         3.0       0.50      0.91      0.65       128
         4.0       0.98      0.99      0.99      1286

    accuracy                           0.98     17511
   macro avg       0.86      0.93      0.88     17511
weighted avg       0.98      0.98      0.98     17511



In [40]:
acc_no_att, f1_no_att = evaluate(model_no_att, X_val_ts, y_val, "NO ATTENTION")

548/548 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step

========== NO ATTENTION ==========
Accuracy: 0.9845240134772428
Macro F1: 0.915949586391412

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99     14494
         1.0       0.95      0.75      0.84       445
         2.0       0.98      0.94      0.96      1158
         3.0       0.77      0.84      0.80       128
         4.0       0.99      0.99      0.99      1286

    accuracy                           0.98     17511
   macro avg       0.94      0.90      0.92     17511
weighted avg       0.98      0.98      0.98     17511



In [41]:
acc_no_weights, f1_no_weights = evaluate(model_no_weights, X_val_ts, y_val, "NO CLASS WEIGHTS")

548/548 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step

========== NO CLASS WEIGHTS ==========
Accuracy: 0.9849808691679516
Macro F1: 0.9103876964726917

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99     14494
         1.0       0.92      0.77      0.84       445
         2.0       0.97      0.95      0.96      1158
         3.0       0.80      0.74      0.77       128
         4.0       1.00      0.99      0.99      1286

    accuracy                           0.98     17511
   macro avg       0.93      0.89      0.91     17511
weighted avg       0.98      0.98      0.98     17511



In [42]:
import pandas as pd

results = pd.DataFrame({

    "Model":[
        "Raw only",
        "Raw + Smoothed",
        "Raw + Derivative",
        "Full model",
        "Without attention",
        "Without class weighting"
    ],

    "Accuracy":[
        acc_raw,
        acc_rs,
        acc_rd,
        acc_full,
        acc_no_att,
        acc_no_weights
    ],

    "Macro F1":[
        f1_raw,
        f1_rs,
        f1_rd,
        f1_full,
        f1_no_att,
        f1_no_weights
    ]
})

print("\n===== ABLATION RESULTS =====")
print(results)


===== ABLATION RESULTS =====
                     Model  Accuracy  Macro F1
0                 Raw only  0.977500  0.891338
1           Raw + Smoothed  0.981897  0.896882
2         Raw + Derivative  0.986237  0.927923
3               Full model  0.979441  0.881416
4        Without attention  0.984524  0.915950
5  Without class weighting  0.984981  0.910388


In [43]:

import numpy as np

# predictions
pred_full = np.argmax(model_full.predict(X_val_ts), axis=1)
pred_no_att = np.argmax(model_no_att.predict(X_val_ts), axis=1)
pred_raw = np.argmax(model_raw.predict(X_val_raw), axis=1)
pred_no_weights = np.argmax(model_no_weights.predict(X_val_ts), axis=1)

# if you trained raw+smooth
pred_rs = np.argmax(model_rs.predict(X_val_rs), axis=1)

548/548 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step
548/548 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step
548/548 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step
548/548 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step
548/548 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step


In [44]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

def run_mcnemar(y_true, pred1, pred2, name):

    both_correct = np.sum((pred1 == y_true) & (pred2 == y_true))
    model1_correct = np.sum((pred1 == y_true) & (pred2 != y_true))
    model2_correct = np.sum((pred1 != y_true) & (pred2 == y_true))
    both_wrong = np.sum((pred1 != y_true) & (pred2 != y_true))

    table = [[both_correct, model1_correct],
             [model2_correct, both_wrong]]

    result = mcnemar(table, exact=False, correction=True)

    print("\nMcNemar Test:", name)
    print("Statistic:", result.statistic)
    print("p-value:", result.pvalue)

    if result.pvalue < 0.05:
        print("Result: Statistically significant")
    else:
        print("Result: Not statistically significant")

In [45]:
run_mcnemar(y_val, pred_full, pred_no_att, "Full vs No Attention")
run_mcnemar(y_val, pred_full, pred_raw, "Full vs Raw")
run_mcnemar(y_val, pred_full, pred_no_weights, "Full vs No Class Weights")
run_mcnemar(y_val, pred_full, pred_rs, "Full vs Raw+Smooth")


McNemar Test: Full vs No Attention
Statistic: 23.255255255255257
p-value: 1.418622760145553e-06
Result: Statistically significant

McNemar Test: Full vs Raw
Statistic: 2.6691176470588234
p-value: 0.10231273169614764
Result: Not statistically significant

McNemar Test: Full vs No Class Weights
Statistic: 29.633440514469452
p-value: 5.219663567798859e-08
Result: Statistically significant

McNemar Test: Full vs Raw+Smooth
Statistic: 4.806539509536785
p-value: 0.028351924127817556
Result: Statistically significant


In [46]:
from scipy.stats import ttest_rel

def run_ttest(y_true, pred1, pred2, name):

    correct1 = (pred1 == y_true).astype(int)
    correct2 = (pred2 == y_true).astype(int)

    t_stat, p = ttest_rel(correct1, correct2)

    print("\nPaired t-test:", name)
    print("t-statistic:", t_stat)
    print("p-value:", p)

In [47]:
run_ttest(y_val, pred_full, pred_no_att, "Full vs No Attention")
run_ttest(y_val, pred_full, pred_raw, "Full vs Raw")
run_ttest(y_val, pred_full, pred_no_weights, "Full vs No Class Weights")
run_ttest(y_val, pred_full, pred_rs, "Full vs Raw+Smooth")


Paired t-test: Full vs No Attention
t-statistic: -4.880346530671316
p-value: 1.0683147452937098e-06

Paired t-test: Full vs Raw
t-statistic: 1.6833389499076687
p-value: 0.0923273205639312

Paired t-test: Full vs No Class Weights
t-statistic: -5.5049658834665545
p-value: 3.744268813978752e-08

Paired t-test: Full vs Raw+Smooth
t-statistic: -2.2448405960931934
p-value: 0.0247907991392877


In [48]:
from scipy.stats import wilcoxon

def run_wilcoxon(y_true, pred1, pred2, name):

    correct1 = (pred1 == y_true).astype(int)
    correct2 = (pred2 == y_true).astype(int)

    stat, p = wilcoxon(correct1, correct2)

    print("\nWilcoxon Test:", name)
    print("Statistic:", stat)
    print("p-value:", p)

In [49]:
run_wilcoxon(y_val, pred_full, pred_no_att, "Full vs No Attention")
run_wilcoxon(y_val, pred_full, pred_raw, "Full vs Raw")
run_wilcoxon(y_val, pred_full, pred_no_weights, "Full vs No Class Weights")
run_wilcoxon(y_val, pred_full, pred_rs, "Full vs Raw+Smooth")


Wilcoxon Test: Full vs No Attention
Statistic: 20374.0
p-value: 1.0761870482107276e-06

Wilcoxon Test: Full vs Raw
Statistic: 38241.5
p-value: 0.09232654595631032

Wilcoxon Test: Full vs No Class Weights
Statistic: 16692.0
p-value: 3.790050326892583e-08

Wilcoxon Test: Full vs Raw+Smooth
Statistic: 29808.0
p-value: 0.024794996770531104
